# 싱글턴 vs 멀티턴 호출 차이

OpenAI Responses API에서 독립 호출과 이어지는 대화 호출의 차이를 확인합니다.


## 0. 준비


In [1]:
import os

from dotenv import load_dotenv
from openai import APIError, BadRequestError, OpenAI, RateLimitError

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 먼저 설정하세요.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI(api_key=api_key)

print(f"사용 모델: {MODEL}")


사용 모델: gpt-5.4-mini


In [2]:
def safe_create(label: str, **kwargs):
    print(f"\n--- {label} ---")
    try:
        response = client.responses.create(model=MODEL, **kwargs)
    except BadRequestError as exc:
        print("BadRequestError:", getattr(exc, "message", str(exc)))
        return None
    except RateLimitError as exc:
        print("RateLimitError:", exc)
        return None
    except APIError as exc:
        print("APIError:", exc)
        return None

    print(response.output_text)
    if getattr(response, "usage", None):
        print(
            f"\n토큰: input={response.usage.input_tokens}, "
            f"output={response.usage.output_tokens}, total={response.usage.total_tokens}"
        )
    return response


## 1. 싱글턴 호출

각 요청이 독립적으로 처리됩니다. 두 번째 요청은 첫 번째 응답을 자동으로 기억하지 않습니다.


In [3]:
question = "파이썬이 뭐야? 한 문장으로 답해줘."

single_first = safe_create(
    "싱글턴 1",
    input=question,
    max_output_tokens=120,
)



--- 싱글턴 1 ---
파이썬은 배우기 쉽고 다양한 작업에 쓰이는 인기 있는 프로그래밍 언어입니다.

토큰: input=22, output=27, total=49


In [4]:
question = "방금 말한 것보다 더 쉽게 설명해줘."

single_second = safe_create(
    "싱글턴 2",
    input=question,
    max_output_tokens=120,
)



--- 싱글턴 2 ---
물론이야.  
근데 **어떤 내용**을 더 쉽게 설명해주면 될지 아직 안 보여.

**방금 말한 내용**을 그대로 붙여주면,  
내가 **아주 쉽게, 짧게** 다시 설명해줄게.

토큰: input=18, output=62, total=80


## 2. `previous_response_id`로 멀티턴 호출

이전 응답의 `id`를 넘기면 다음 요청에서 앞 대화를 이어갈 수 있습니다.


In [5]:
turn1 = safe_create(
    "멀티턴 1",
    input="파이썬이 뭐야? 한 문장으로 답해줘.",
    max_output_tokens=120,
)



--- 멀티턴 1 ---
파이썬은 사람이 읽기 쉬운 문법을 가진, 다양한 일을 자동화하고 프로그램을 만들 수 있는 프로그래밍 언어입니다.

토큰: input=22, output=36, total=58


In [6]:
turn2 = None

if turn1 is not None:
    turn2 = safe_create(
        "멀티턴 2",
        previous_response_id=turn1.id,
        input="방금 말한 것보다 더 쉽게 설명해줘.",
        max_output_tokens=160,
    )



--- 멀티턴 2 ---
파이썬은 컴퓨터에게 시키고 싶은 일을 적는 쉬운 말이에요.

토큰: input=76, output=25, total=101


In [7]:
turn3 = None

if turn2 is not None:
    turn3 = safe_create(
        "멀티턴 3",
        previous_response_id=turn2.id,
        input="예시 코드도 짧게 줘.",
        max_output_tokens=220,
    )



--- 멀티턴 3 ---
물론이죠. 아주 짧게 예시를 들면:

```python
print("안녕!")
```

이 코드는 컴퓨터가 **안녕!** 이라고 화면에 보여주게 해요.

토큰: input=117, output=50, total=167


## 3. 지시문 + 멀티턴


In [8]:
tutor_first = safe_create(
    "튜터 대화 1",
    instructions=(
        "당신은 초보자를 가르치는 Python 튜터입니다. "
        "답변은 짧고 친절하게 작성하고, 마지막에 확인 질문을 붙이세요."
    ),
    input="리스트가 뭐야?",
    max_output_tokens=180,
)

tutor_second = None
if tutor_first is not None:
    tutor_second = safe_create(
        "튜터 대화 2",
        previous_response_id=tutor_first.id,
        input="딕셔너리랑 차이가 뭔데?",
        max_output_tokens=220,
    )



--- 튜터 대화 1 ---
리스트는 **여러 값을 순서대로 담는 자료형**이에요.  
파이썬에서 대괄호 `[]`로 만들어요.

예시:
```python
fruits = ["사과", "바나나", "딸기"]
```

특징:
- **순서가 있음**
- **값을 추가/삭제**할 수 있음
- **서로 다른 자료형**도 함께 넣을 수 있음

예:
```python
data = [1, "hi", 3.5]
```

원하면 제가 **리스트 인덱스, 추가, 삭제**도 바로 설명해드릴까요?

토큰: input=52, output=144, total=196

--- 튜터 대화 2 ---
리스트와 딕셔너리는 둘 다 여러 값을 담지만, **저장 방식이 달라요**.

### 리스트
- **순서대로** 저장
- 값을 꺼낼 때 **인덱스(0, 1, 2...)** 를 사용
- 예:
```python
fruits = ["사과", "바나나", "딸기"]
print(fruits[0])  # 사과
```

### 딕셔너리
- **키(key)와 값(value)** 쌍으로 저장
- 값을 꺼낼 때 **키 이름**을 사용
- 예:
```python
person = {"name": "철수", "age": 20}
print(person["name"])  # 철수
```

### 차이 한 줄 요약
- **리스트**: “몇 번째?”
- **딕셔너리**: “무엇의 값?”

### 언제 쓰나?
- **리스트**: 순서가 중요할 때, 여러 항

토큰: input=174, output=220, total=394


## 4. 수동 기록 관리

`previous_response_id` 대신 메시지 목록을 직접 관리할 수도 있습니다.
작은 실습에서는 편하지만, 기록이 길어질수록 토큰 비용이 늘어납니다.


In [9]:
manual_messages = [
    {"role": "developer", "content": "답변은 초보자에게 맞추고 4문장 이내로 작성하세요."}
]


def manual_turn(user_message: str):
    manual_messages.append({"role": "user", "content": user_message})
    response = safe_create(
        f"수동 기록: {user_message}",
        input=manual_messages,
        max_output_tokens=180,
    )
    if response is not None:
        manual_messages.append({"role": "assistant", "content": response.output_text})
    return response


manual_turn("제일 먼저 배워야 할 Python 문법은 뭐야?")



--- 수동 기록: 제일 먼저 배워야 할 Python 문법은 뭐야? ---
제일 먼저는 **변수와 자료형**, 그리고 **출력(print)** 을 배우는 게 좋아요.  
그다음은 **if 조건문**, **for/while 반복문**, **함수** 순서로 익히면 됩니다.  
이 4가지만 알아도 간단한 프로그램을 바로 만들 수 있어요.  
원하시면 제가 **하루 학습 순서**로 아주 쉽게 정리해드릴게요.

토큰: input=44, output=100, total=144


Response(id='resp_087c02d8a7d77252006a4726760d70819abd4a913fc29cc8e4', created_at=1783047798.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_087c02d8a7d77252006a47267650c8819ab53c61e1ea263580', content=[ResponseOutputText(annotations=[], text='제일 먼저는 **변수와 자료형**, 그리고 **출력(print)** 을 배우는 게 좋아요.  \n그다음은 **if 조건문**, **for/while 반복문**, **함수** 순서로 익히면 됩니다.  \n이 4가지만 알아도 간단한 프로그램을 바로 만들 수 있어요.  \n원하시면 제가 **하루 학습 순서**로 아주 쉽게 정리해드릴게요.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1783047799.0, conversation=None, max_output_tokens=180, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_

In [10]:
manual_turn("왜 그 문법부터 배우는 게 좋아?")

print(f"현재 manual_messages 길이: {len(manual_messages)}")



--- 수동 기록: 왜 그 문법부터 배우는 게 좋아? ---
변수와 출력은 **값을 저장하고 보여주는 가장 기본**이라서, Python이 어떻게 동작하는지 바로 감을 잡기 좋아요.  
if와 반복문은 **프로그램이 판단하고 여러 번 일하는 방법**이라서, 간단한 문제를 스스로 풀 수 있게 해줍니다.  
함수는 **코드를 정리하고 여러 번 재사용하는 방법**이라서, 나중에 더 큰 프로그램을 만들 때 꼭 필요해요.  
그래서 이 순서가 **쉽고 자연스럽게 실력으로 이어지기** 때문에 초보자에게 가장 좋습니다.

토큰: input=160, output=136, total=296
현재 manual_messages 길이: 5


In [11]:
print("--- 현재 수동 대화 기록 ---")
for index, message in enumerate(manual_messages, start=1):
    content = message["content"].replace("\n", " ")
    print(f"{index}. {message['role']}: {content[:120]}")


--- 현재 수동 대화 기록 ---
1. developer: 답변은 초보자에게 맞추고 4문장 이내로 작성하세요.
2. user: 제일 먼저 배워야 할 Python 문법은 뭐야?
3. assistant: 제일 먼저는 **변수와 자료형**, 그리고 **출력(print)** 을 배우는 게 좋아요.   그다음은 **if 조건문**, **for/while 반복문**, **함수** 순서로 익히면 됩니다.   이 4가지만 알아
4. user: 왜 그 문법부터 배우는 게 좋아?
5. assistant: 변수와 출력은 **값을 저장하고 보여주는 가장 기본**이라서, Python이 어떻게 동작하는지 바로 감을 잡기 좋아요.   if와 반복문은 **프로그램이 판단하고 여러 번 일하는 방법**이라서, 간단한 문제를 스스로


## 5. 스트리밍 응답

긴 답변을 기다리지 않고 생성되는 텍스트를 조금씩 받을 수 있습니다.


In [12]:
def stream_openai(label: str, **kwargs):
    print(f"\n--- {label} ---")
    try:
        stream = client.responses.create(
            model=MODEL,
            stream=True,
            **kwargs,
        )
        for event in stream:
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
        print()
    except BadRequestError as exc:
        print("BadRequestError:", getattr(exc, "message", str(exc)))
    except APIError as exc:
        print("APIError:", exc)


In [13]:
stream_openai(
    "스트리밍 단일 응답",
    input="Python API 수업에서 스트리밍이 필요한 상황을 3가지로 설명해줘.",
    max_output_tokens=250,
)



--- 스트리밍 단일 응답 ---
Python API 수업에서 **스트리밍(streaming)** 이 필요한 상황은 보통 “한 번에 다 받거나 보내기엔 데이터가 너무 크거나, 실시간성이 중요할 때”입니다. 대표적으로 3가지는 아래와 같습니다.

1. **대용량 파일 전송**
   - 예: 동영상, 이미지 묶음, 로그 파일, 백업 파일 업/다운로드
   - 한 번에 메모리로 다 올리면 부담이 크므로, 작은 단위로 나눠서 전송하면 효율적입니다.

2. **실시간 데이터 처리**
   - 예: 채팅 메시지, 주식 시세, 센서 데이터, 게임 이벤트
   - 전체가 끝날 때까지 기다리지 않고, 데이터가 들어오는 즉시 처리해야 하므로 스트리밍이 유리합니다.

3. **긴 작업의 진행 상황을 즉시 전달할 때**
   - 예: AI 모델 생성 응답, 검색 결과를 순차적으로 보여주기, 서버에서 계산 중간 결과 전달
   - 사용자가 결과를 한 번에 받기보다, 중간부터 조금씩 보면서 기다릴 수 있어 사용자


In [14]:
if turn3 is not None:
    stream_openai(
        "스트리밍 멀티턴 응답",
        previous_response_id=turn3.id,
        input="지금까지의 설명을 한 문단으로 요약해줘.",
        max_output_tokens=180,
    )



--- 스트리밍 멀티턴 응답 ---
파이썬은 컴퓨터에게 하고 싶은 일을 쉽게 적을 수 있는 프로그래밍 언어이며, 사람이 읽기 쉬운 문법을 가지고 있어서 배우기 비교적 쉽습니다. 예를 들어 `print("안녕!")`이라고 쓰면 컴퓨터가 화면에 **안녕!**이라고 보여줍니다.


## 6. 컨텍스트 성장 확인


In [15]:
context_questions = [
    "API 키를 안전하게 보관하는 방법을 한 문장으로 알려줘.",
    "방금 답변에서 가장 중요한 규칙 하나만 골라줘.",
    "그 규칙을 어겼을 때 생기는 문제를 예시로 들어줘.",
]

previous_id = None

for index, question in enumerate(context_questions, start=1):
    params = {
        "input": question,
        "max_output_tokens": 160,
    }
    if previous_id is not None:
        params["previous_response_id"] = previous_id

    response = safe_create(f"컨텍스트 체인 {index}", **params)
    if response is None:
        break
    previous_id = response.id



--- 컨텍스트 체인 1 ---
API 키는 코드에 직접 넣지 말고 환경 변수나 비밀 관리 서비스에 저장해 안전하게 보관하세요.

토큰: input=23, output=30, total=53

--- 컨텍스트 체인 2 ---
코드에 직접 넣지 말고 저장하세요.

토큰: input=74, output=15, total=89

--- 컨텍스트 체인 3 ---
예를 들어 API 키를 코드에 직접 적어 GitHub에 올리면, 다른 사람이 그 키를 복사해 내 계정으로 무단 요청을 보내 요금이 발생하거나 서비스가 악용될 수 있어요.

토큰: input=113, output=53, total=166


## 7. 정리

| 방식 | 장점 | 주의점 |
| --- | --- | --- |
| 싱글턴 | 단순하고 예측하기 쉬움 | 이전 대화를 기억하지 않음 |
| `previous_response_id` | 서버 측 대화 흐름을 쉽게 이어감 | 어떤 응답을 기준으로 잇는지 관리 필요 |
| 수동 기록 | 기록을 직접 편집 가능 | 기록이 길어지면 토큰 비용 증가 |
| 스트리밍 | 긴 응답을 빠르게 보여줄 수 있음 | 이벤트 처리 코드가 필요 |
